# Hull Tactical Market Prediction – Ridge Regression Model
This notebook demonstrates how Ridge Regression was applied to predict financial forward returns using Hull Tactical’s dataset.  
We implement data preprocessing, model training, hyperparameter tuning, and evaluation — all step by step.

## Import Libraries

In [196]:
import pandas as pd
from sklearn.linear_model import Ridge
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error
from sklearn.linear_model import RidgeCV

## Load, Explore, and Clean the Data

In [199]:
# Load datasets
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")

# Preview the data
display(train.head())
train.dropna(inplace=True)
display(train.head())

,date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,V3,V4,V5,V6,V7,V8,V9,forward_returns,risk_free_rate,market_forward_excess_returns
0,0,0,0,0,1,1,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.002421,0.000301,-0.003038
1,1,0,0,0,1,1,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.008495,0.000303,-0.009114
2,2,0,0,0,1,0,0,0,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.009624,0.000301,-0.010243
3,3,0,0,0,1,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.004662,0.000299,0.004046
4,4,0,0,0,1,0,0,0,0,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.011686,0.000299,-0.012301


,date_id,D1,D2,D3,D4,D5,D6,D7,D8,D9,...,V3,V4,V5,V6,V7,V8,V9,forward_returns,risk_free_rate,market_forward_excess_returns
6969,6969,0,0,0,0,0,-1,0,0,0,...,0.976852,0.996693,0.534651,0.884921,-0.911387,0.979167,-0.847754,0.001145,0.000040,0.000797
6970,6970,0,0,0,0,0,0,0,1,0,...,0.986772,0.997354,0.283256,0.769180,-0.842354,0.970238,-0.821913,0.004738,0.000040,0.004390
6971,6971,0,0,0,0,1,0,0,1,0,...,0.962963,0.999339,0.713303,0.814153,-0.933089,0.951720,-0.817906,0.006016,0.000040,0.005669
6972,6972,0,0,0,0,1,0,1,1,0,...,0.946429,1.000000,0.583019,0.809524,-1.051731,0.953042,-0.887412,0.001414,0.000039,0.001067
6973,6973,0,0,0,0,1,0,0,0,1,...,0.960979,0.999339,1.054972,0.567460,-1.138938,0.951058,-0.930502,-0.007182,0.000039,-0.007529


## Feature and Target Separation

In [202]:
# Separate features (X) and target (Y)
X = train.drop(columns=["forward_returns"])
y = train["forward_returns"]

In [204]:
print("Target mean:", y.mean())
print("Target std:", y.std())

Target mean: 0.000627481530872643
Target std: 0.010800718859045317


## Train–Validation Split

In [207]:
X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.2, random_state=42)

## Feature Scaling

In [210]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

## Baseline Ridge Regression Model

In [213]:
ridge = Ridge(alpha=1.0)
ridge.fit(X_train_scaled, y_train)

y_pred = ridge.predict(X_val_scaled)
mse = mean_squared_error(y_val, y_pred)
print("Validation MSE:", mse)

Validation MSE: 2.1324839916823963e-09


## Experimenting with Alpha Values

In [216]:
for a in [0.01, 0.1, 1, 10, 100]:
    ridge = Ridge(alpha=a)
    ridge.fit(X_train_scaled, y_train)
    preds = ridge.predict(X_val_scaled)
    mse = mean_squared_error(y_val, preds)
    print(f"alpha={a}, MSE={mse:.10f}")

alpha=0.01, MSE=0.0000000022
alpha=0.1, MSE=0.0000000022
alpha=1, MSE=0.0000000021
alpha=10, MSE=0.0000000069
alpha=100, MSE=0.0000004549


## Compare first few predictions with actual values

In [219]:
comparison = pd.DataFrame({"Actual": y_val.values[:10], "Predicted": y_pred[:10]})
display(comparison)

,Actual,Predicted
0,0.009239,0.009235
1,-0.003696,-0.003704
2,0.010211,0.010215
3,-0.007572,-0.007564
4,0.008619,0.008596
5,-0.002328,-0.002322
6,0.011558,0.011529
7,0.009072,0.009056
8,0.023791,0.023739
9,0.014557,0.014526


## Cross-Validation with RidgeCV

In [222]:
ridge_cv = RidgeCV(alphas=[0.001, 0.01, 0.1, 1, 10, 100], cv=5, scoring='neg_mean_squared_error')
ridge_cv.fit(X_train_scaled, y_train)

print("Best alpha from cross-validation:", ridge_cv.alpha_)

Best alpha from cross-validation: 0.1


## Model Evaluation

In [225]:
y_val_pred_cv = ridge_cv.predict(X_val_scaled)
mse_cv = mean_squared_error(y_val, y_val_pred_cv)

In [227]:
r2_simple = ridge.score(X_val_scaled, y_val)
r2_cv = ridge_cv.score(X_val_scaled, y_val)

In [229]:
print("Validation MSE (CV model):", mse_cv)
print("Baseline R² (alpha=1.0):", r2_simple)
print("CV Model R²:", r2_cv)

Validation MSE (CV model): 2.1639868079370684e-09
Baseline R² (alpha=1.0): 0.9960878289498777
CV Model R²: 0.9999813905113445


## Feature Importance Analysis

In [232]:
coef_df = pd.DataFrame({"Feature": X.columns, "Coefficient": ridge_cv.coef_}).sort_values(by="Coefficient", ascending=False)
print("\nTop 10 features by coefficient:")
display(coef_df.head(10))


Top 10 features by coefficient:


,Feature,Coefficient
96,market_forward_excess_returns,0.010801
95,risk_free_rate,0.000081
94,V9,0.000034
83,V10,0.000030
21,E2,0.000022
41,M11,0.000019
80,S8,0.000015
68,P8,0.000014
39,M1,0.000014
66,P6,0.000012


## Align Train and Test Features

In [235]:
common_cols = X.columns.intersection(test.columns)
X_train = X_train[common_cols]
X_val = X_val[common_cols]
test = test[common_cols]

In [237]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)

## Final Model Training and Test Predictions

In [240]:
ridge_cv.fit(X_train_scaled, y_train)

X_test_scaled = scaler.transform(test)
test_preds = ridge_cv.predict(X_test_scaled)

## Save Predictions

In [243]:
submission = pd.DataFrame({"Id": test.index, "Predicted": test_preds})
submission.to_csv("ridge_predictions.csv", index=False)

In [245]:
print("Predictions saved to ridge_predictions.csv")

Predictions saved to ridge_predictions.csv


## Summary and Next Steps

**Summary:**
- Best alpha value: 0.1
- Validation R²: 0.9999813905113445
- Top predictive features: market_forward_excess_returns, risk_free_rate, V9